In [1]:
# 필요한 라이브러리 import
!pip install torch-geometric
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.datasets import FB15k_237
import random
import copy
import time
from typing import List, Tuple, Set, Optional
from tqdm import tqdm  # 진행 상황 표시용 (선택)

print("PyTorch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\SAMSUNG\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
C:\Users\SAMSUNG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch 버전: 2.9.1+cpu
CUDA 사용 가능: False


In [2]:
# TransE 모델 클래스 정의
class TransE(nn.Module):
    def __init__(self, num_entities: int, num_relations: int, 
                dim: int = 50, margin: float = 1.0, norm_p: int = 1):
        super().__init__()
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)

        # 논문 초기화: uniform(-6/√d, 6/√d)
        bound = 6.0 / dim**0.5
        nn.init.uniform_(self.entity_emb.weight, -bound, bound)
        nn.init.uniform_(self.relation_emb.weight, -bound, bound)

        self._normalize_entities()
        self.margin = margin
        self.norm_p = norm_p

    def _normalize_entities(self):
        """엔티티 임베딩을 L2 norm=1로 정규화 (논문 필수)"""
        with torch.no_grad():
            self.entity_emb.weight[:] = F.normalize(self.entity_emb.weight, p=2, dim=1)

    def score(self, h, r, t):
        """||h + r - t||_p 계산"""
        h_emb = self.entity_emb(h)
        r_emb = self.relation_emb(r)
        t_emb = self.entity_emb(t)
        return torch.norm(h_emb + r_emb - t_emb, p=self.norm_p, dim=1)

    def forward(self, pos_h, pos_r, pos_t, neg_h, neg_r, neg_t):
        """Margin-based ranking loss"""
        pos = self.score(pos_h, pos_r, pos_t)
        neg = self.score(neg_h, neg_r, neg_t)
        return torch.relu(self.margin + pos - neg).mean()

In [3]:
# Raw Mean Rank 계산 함수 (validation용)
def raw_mean_rank(model: TransE, triples: torch.Tensor, device: torch.device,
                triples_set: Set[Tuple[int, int, int]]) -> float:
    """Raw Mean Rank 계산 (논문 validation 기준)"""
    model.eval()
    ranks = []
    with torch.no_grad():
        for h, r, t in triples:
            candidates = torch.arange(model.entity_emb.num_embeddings, device=device)

            # head 예측 (tail 고정)
            scores = model.score(candidates, r.repeat(len(candidates)), t.repeat(len(candidates)))
            rank = (scores < scores[h]).sum().item() + 1
            ranks.append(rank)

            # tail 예측 (head 고정)
            scores = model.score(h.repeat(len(candidates)), r.repeat(len(candidates)), candidates)
            rank = (scores < scores[t]).sum().item() + 1
            ranks.append(rank)

    return sum(ranks) / len(ranks)

In [4]:
# 학습 함수 정의 (train_transE)
def train_transE(
    train_triples: List[Tuple[int, int, int]],
    valid_triples: torch.Tensor,
    num_entities: int,
    num_relations: int,
    dim: int = 50,
    margin: float = 1.0,
    lr: float = 0.001,
    norm_p: int = 1,
    batch_size: int = 128,
    max_epochs: int = 1000,
    patience: int = 50,
    valid_check_interval: int = 20,
    device: Optional[str] = None
) -> TransE:
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    device = torch.device(device)

    model = TransE(num_entities, num_relations, dim=dim, margin=margin, norm_p=norm_p)
    model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=lr)

    triples_set = set(train_triples)
    best_valid_rank = float('inf')
    best_model = None
    counter = 0
    start_time = time.time()

    print(f"Start training | dim={dim} | margin={margin} | norm=L{norm_p} | lr={lr} | device={device}")

    for epoch in range(1, max_epochs + 1):
        model.train()
        random.shuffle(train_triples)
        total_loss = 0.0

        for start in range(0, len(train_triples), batch_size):
            batch = train_triples[start:start + batch_size]
            pos_h = torch.tensor([x[0] for x in batch], device=device)
            pos_r = torch.tensor([x[1] for x in batch], device=device)
            pos_t = torch.tensor([x[2] for x in batch], device=device)

            neg_h = pos_h.clone()
            neg_t = pos_t.clone()

            for i in range(len(batch)):
                h, r, t = batch[i]
                if random.random() < 0.5:
                    while True:
                        cand = random.randint(0, num_entities - 1)
                        if (cand, r, t) not in triples_set:
                            neg_h[i] = cand
                            break
                else:
                    while True:
                        cand = random.randint(0, num_entities - 1)
                        if (h, r, cand) not in triples_set:
                            neg_t[i] = cand
                            break

            optimizer.zero_grad()
            loss = model(pos_h, pos_r, pos_t, neg_h, pos_r, neg_t)
            loss.backward()
            optimizer.step()

            model._normalize_entities()

            total_loss += loss.item() * len(batch)

        avg_loss = total_loss / len(train_triples)
        print(f"Epoch {epoch:4d} | loss {avg_loss:.4f}", end="")

        if epoch % valid_check_interval == 0 or epoch == 1:
            valid_rank = raw_mean_rank(model, valid_triples, device, triples_set)
            print(f" | Valid Raw Mean Rank: {valid_rank:.2f} | elapsed: {time.time() - start_time:.1f}s")

            if valid_rank < best_valid_rank:
                best_valid_rank = valid_rank
                best_model = copy.deepcopy(model)
                counter = 0
            else:
                counter += 1
                if counter >= patience:
                    print(f"Early stopping at epoch {epoch}")
                    break
        else:
            print("")

    print(f"Training finished. Best valid raw mean rank: {best_valid_rank:.2f}")
    return best_model

In [5]:
# 셀 5 전체 교체용 코드 (문자열 ID → 숫자 ID 매핑 추가)
import os
import pandas as pd
from collections import defaultdict

root = './data/FB15k237'
train_path = os.path.join(root, 'raw', 'train.txt')
valid_path = os.path.join(root, 'raw', 'valid.txt')
# test_path = os.path.join(root, 'raw', 'test.txt')  # 필요시

def load_and_map_triples(paths):
    entity2id = {}
    relation2id = {}
    ent_counter = 0
    rel_counter = 0
    triples = []

    for path in paths:
        df = pd.read_csv(path, sep='\t', header=None, names=['h', 'r', 't'])
        for h, r, t in df.values:
            # 엔티티 매핑
            if h not in entity2id:
                entity2id[h] = ent_counter
                ent_counter += 1
            if t not in entity2id:
                entity2id[t] = ent_counter
                ent_counter += 1
            # 관계 매핑
            if r not in relation2id:
                relation2id[r] = rel_counter
                rel_counter += 1
            # 숫자 ID로 변환해서 저장
            triples.append((entity2id[h], relation2id[r], entity2id[t]))

    num_entities = len(entity2id)
    num_relations = len(relation2id)
    
    return triples, num_entities, num_relations, entity2id, relation2id

# train + valid 합쳐서 매핑 (valid에 새로운 엔티티/관계가 있을 수 있음)
all_paths = [train_path, valid_path]
train_triples, num_entities, num_relations, entity2id, relation2id = load_and_map_triples(all_paths)

# valid만 따로 다시 로드해서 숫자 ID로 변환 (매핑 재사용)
valid_df = pd.read_csv(valid_path, sep='\t', header=None, names=['h', 'r', 't'])
valid_triples_list = []
for h, r, t in valid_df.values:
    # 매핑에 없는 경우는 거의 없지만 대비
    h_id = entity2id.get(h, -1)
    r_id = relation2id.get(r, -1)
    t_id = entity2id.get(t, -1)
    if h_id != -1 and r_id != -1 and t_id != -1:
        valid_triples_list.append((h_id, r_id, t_id))
valid_triples = torch.tensor(valid_triples_list)

print(f"Entities: {num_entities:,} | Relations: {num_relations:,}")
print(f"Train triples: {len(train_triples):,} | Valid triples: {len(valid_triples):,}")

# negative sampling용 set (숫자 ID 버전)
triples_set = set(train_triples)

Entities: 14,513 | Relations: 237
Train triples: 289,650 | Valid triples: 17,535


In [6]:
best_model = train_transE(
    train_triples=train_triples,
    valid_triples=valid_triples,
    num_entities=num_entities,
    num_relations=num_relations,
    dim=50,
    margin=1.0,
    lr=0.01,
    norm_p=1,
    batch_size=128,           # GPU 메모리에 따라 256~512로 늘려도 됨
    max_epochs=1000,
    patience=50,
    valid_check_interval=200
)

Start training | dim=50 | margin=1.0 | norm=L1 | lr=0.01 | device=cpu
Epoch    1 | loss 1.1135 | Valid Raw Mean Rank: 6527.09 | elapsed: 90.7s
Epoch    2 | loss 1.0551
Epoch    3 | loss 1.0180
Epoch    4 | loss 0.9886
Epoch    5 | loss 0.9641
Epoch    6 | loss 0.9475
Epoch    7 | loss 0.9285
Epoch    8 | loss 0.9121
Epoch    9 | loss 0.8957
Epoch   10 | loss 0.8845
Epoch   11 | loss 0.8715
Epoch   12 | loss 0.8622
Epoch   13 | loss 0.8510
Epoch   14 | loss 0.8400
Epoch   15 | loss 0.8301
Epoch   16 | loss 0.8183
Epoch   17 | loss 0.8098
Epoch   18 | loss 0.8012
Epoch   19 | loss 0.7921
Epoch   20 | loss 0.7822
Epoch   21 | loss 0.7767
Epoch   22 | loss 0.7716
Epoch   23 | loss 0.7607
Epoch   24 | loss 0.7548
Epoch   25 | loss 0.7463
Epoch   26 | loss 0.7419
Epoch   27 | loss 0.7343
Epoch   28 | loss 0.7285
Epoch   29 | loss 0.7245
Epoch   30 | loss 0.7134
Epoch   31 | loss 0.7109
Epoch   32 | loss 0.7030
Epoch   33 | loss 0.6953
Epoch   34 | loss 0.6866
Epoch   35 | loss 0.6830
Epoch  

In [7]:
# 모델 저장 (필요할 때)
torch.save(best_model.state_dict(), "transe_fb15k237_dim50_margin1.pth")
print("Best model saved.")

Best model saved.


In [8]:
# 저장된 모델 불러오기
loaded_model = TransE(num_entities, num_relations, dim=50, margin=1.0, norm_p=1)
loaded_model.load_state_dict(torch.load("transe_fb15k237_dim50_margin1.pth"))
loaded_model.to('cuda' if torch.cuda.is_available() else 'cpu')
loaded_model.eval()  # 예측 모드로 전환

print("모델 불러오기 완료!")

모델 불러오기 완료!


In [9]:
# 예시: 특정 엔티티 + 관계 벡터로 tail 예측해보기
with torch.no_grad():
    # FB15k-237에서 entity id와 이름 매핑은 dataset에 없으므로 id로만
    # 예: 임의의 head와 relation id로 tail 후보 top-5 보기
    head_id = 0
    rel_id = 0
    candidates = torch.arange(num_entities, device=best_model.entity_emb.weight.device)
    scores = best_model.score(
        torch.tensor([head_id], device=best_model.entity_emb.weight.device).repeat(num_entities),
        torch.tensor([rel_id], device=best_model.entity_emb.weight.device).repeat(num_entities),
        candidates
    )
    top5_scores, top5_idx = torch.topk(scores, 5, largest=False)  # 낮은 score = 좋은 순
    print("Top 5 predicted tail entity ids:", top5_idx.tolist())
    print("Their scores:", top5_scores.tolist())

Top 5 predicted tail entity ids: [1, 1713, 4494, 5789, 12322]
Their scores: [13.325819969177246, 13.490950584411621, 13.63651180267334, 13.656525611877441, 13.829880714416504]
